# ETL Silver → Gold

Popula o Star Schema no PostgreSQL a partir dos dados limpos do Silver Layer.

## 1. Imports

Bibliotecas necessarias para o ETL.

In [120]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch
import warnings
from sqlalchemy import create_engine

warnings.filterwarnings('ignore', message='.*SQLAlchemy.*')

## 2. Configuracao

Parametros de conexao ao banco de dados.

In [121]:
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'database': 'uber_analytics',
    'user': 'postgres',
    'password': 'postgres'
}

conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

print('Conexao estabelecida com sucesso!')

Conexao estabelecida com sucesso!


## 3. Verificacao Rapida

Verificar se existem dados no Silver.

In [122]:
cur.execute("SELECT COUNT(*) FROM silver.booking")
silver_total = cur.fetchone()[0]
print(f'Total de registros no Silver: {silver_total:,}')

if silver_total == 0:
    print('ALERTA: Nao ha dados no Silver. Execute o ETL raw_to_silver primeiro.')
else:
    print(f'Silver OK: {silver_total:,} registros encontrados.')

Total de registros no Silver: 148,767
Silver OK: 148,767 registros encontrados.


## 4. Executar DDL

Criar schema DW e tabelas (dimensoes + fato).

In [123]:
import os

ddl_path = os.path.join('..', 'Data Layer', 'gold', 'ddl.sql')

with open(ddl_path, 'r', encoding='utf-8') as f:
    ddl_sql = f.read()

cur.execute(ddl_sql)
conn.commit()

print('DDL executado com sucesso! Schema DW criado.')

DDL executado com sucesso! Schema DW criado.


## 5. Carregar Dados do Silver

Ler todos os dados da tabela silver.booking.

In [124]:
query = "SELECT * FROM silver.booking"
df_silver = pd.read_sql(query, conn)

print(f'Dados carregados: {len(df_silver):,} registros')
print(f'Colunas: {list(df_silver.columns)}')

Dados carregados: 148,767 registros
Colunas: ['id', 'date', 'time', 'status', 'customer_id', 'vehicle', 'pickup', 'drop', 'pickup_zone', 'pickup_region', 'drop_zone', 'drop_region', 'vtat', 'ctat', 'cancelled_by_customer', 'reason_cancelled_by_customer', 'cancelled_by_driver', 'reason_cancelled_by_driver', 'incomplete', 'reason_incomplete', 'value', 'distance', 'distance_category', 'value_per_km', 'driver_rating', 'customer_rating', 'payment']


## 6. Dimensao Tempo (dim_tmp)

Criar dimensao de tempo com hierarquia completa.

In [125]:
df_silver['date'] = pd.to_datetime(df_silver['date'])

dim_tmp = df_silver[['date']].drop_duplicates().copy()
dim_tmp.rename(columns={'date': 'data'}, inplace=True)

dim_tmp['ano'] = dim_tmp['data'].dt.year
dim_tmp['mes'] = dim_tmp['data'].dt.month
dim_tmp['dia'] = dim_tmp['data'].dt.day
dim_tmp['dia_semana'] = dim_tmp['data'].dt.dayofweek
dim_tmp['nome_dia_semana'] = dim_tmp['data'].dt.day_name()
dim_tmp['trimestre'] = dim_tmp['data'].dt.quarter
dim_tmp['semana_ano'] = dim_tmp['data'].dt.isocalendar().week
dim_tmp['eh_fim_semana'] = dim_tmp['dia_semana'].isin([5, 6])
dim_tmp['mes_ano'] = dim_tmp['data'].dt.strftime('%Y-%m')
dim_tmp['ano_trimestre'] = dim_tmp['ano'].astype(str) + '-Q' + dim_tmp['trimestre'].astype(str)

def get_periodo_dia(hora):
    if 6 <= hora < 12:
        return 'Manha'
    elif 12 <= hora < 18:
        return 'Tarde'
    elif 18 <= hora < 24:
        return 'Noite'
    else:
        return 'Madrugada'

dim_tmp['periodo_dia'] = df_silver.groupby(df_silver['date'].dt.date)['time'].first().apply(lambda x: get_periodo_dia((x.hour if hasattr(x, 'hour') else int(str(x).split(':')[0])) if pd.notna(x) else 12)).values

def eh_horario_pico(hora):
    return (7 <= hora < 10) or (17 <= hora < 20)

dim_tmp['eh_horario_pico'] = df_silver.groupby(df_silver['date'].dt.date)['time'].first().apply(lambda x: eh_horario_pico((x.hour if hasattr(x, 'hour') else int(str(x).split(':')[0])) if pd.notna(x) else 12)).values

print(f'Dimensao Tempo: {len(dim_tmp)} registros')

Dimensao Tempo: 365 registros


## 7. Dimensao Localizacao (dim_loc)

Criar dimensao de localizacao com origem e destino.

In [126]:
pickup_locs = df_silver[['pickup', 'pickup_zone', 'pickup_region']].copy()
pickup_locs.columns = ['local', 'zona', 'regiao']

drop_locs = df_silver[['drop', 'drop_zone', 'drop_region']].copy()
drop_locs.columns = ['local', 'zona', 'regiao']

dim_loc = pd.concat([pickup_locs, drop_locs], ignore_index=True).drop_duplicates()

def get_tipo_area(zona):
    if 'Airport' in zona:
        return 'Aeroporto'
    elif any(x in zona for x in ['Central', 'Connaught', 'Market']):
        return 'Comercial'
    else:
        return 'Residencial'

dim_loc['tipo_area'] = dim_loc['zona'].apply(get_tipo_area)

print(f'Dimensao Localizacao: {len(dim_loc)} registros')

Dimensao Localizacao: 176 registros


## 8. Dimensao Veiculo (dim_vei)

Criar dimensao de veiculo com categorias.

In [127]:
dim_vei = df_silver[['vehicle']].drop_duplicates().copy()
dim_vei.rename(columns={'vehicle': 'tipo_veiculo'}, inplace=True)

def get_categoria_veiculo(tipo):
    tipo_lower = tipo.lower()
    if any(x in tipo_lower for x in ['go', 'mini', 'auto']):
        return 'Economy'
    elif any(x in tipo_lower for x in ['xl', 'suv', 'premier']):
        return 'Premium'
    elif any(x in tipo_lower for x in ['lux', 'black']):
        return 'Luxury'
    else:
        return 'Standard'

dim_vei['categoria_veiculo'] = dim_vei['tipo_veiculo'].apply(get_categoria_veiculo)

print(f'Dimensao Veiculo: {len(dim_vei)} registros')

Dimensao Veiculo: 7 registros


## 9. Dimensao Cliente (dim_cli)

Criar dimensao de cliente com segmentacao.

In [128]:
cliente_stats = df_silver.groupby('customer_id').agg(
    total_viagens_historico=('id', 'count')
).reset_index()

def get_segmento_cliente(total_viagens):
    if total_viagens < 5:
        return 'Occasional'
    elif total_viagens < 20:
        return 'Regular'
    else:
        return 'VIP'

cliente_stats['segmento_cliente'] = cliente_stats['total_viagens_historico'].apply(get_segmento_cliente)

dim_cli = cliente_stats.copy()
dim_cli.rename(columns={'customer_id': 'id_cliente'}, inplace=True)

print(f'Dimensao Cliente: {len(dim_cli)} registros')

Dimensao Cliente: 147580 registros


## 10. Dimensao Pagamento (dim_pag)

Criar dimensao de pagamento com categorias.

In [129]:
def get_categoria_pagamento(metodo):
    if pd.isna(metodo) or not isinstance(metodo, str):
        return 'Outros'
    
    metodo = str(metodo).lower()
    
    if metodo == 'cash':
        return 'Cash'
    else:
        return 'Digital'

dim_pag = df_silver[['payment']].dropna().drop_duplicates().copy()
dim_pag = dim_pag.rename(columns={'payment': 'metodo_pagamento'})
dim_pag['categoria_pagamento'] = dim_pag['metodo_pagamento'].apply(get_categoria_pagamento)

print(f'dim_pag preparada: {len(dim_pag)} registros')

dim_pag preparada: 5 registros


## 11. Inserir Dimensoes no Banco

Carregar todas as dimensoes no schema DW.

In [130]:
engine = create_engine(f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

dim_tmp.to_sql('dim_tmp', engine, schema='dw', if_exists='append', index=False)
print(f'dim_tmp: {len(dim_tmp)} registros inseridos')

dim_loc.to_sql('dim_loc', engine, schema='dw', if_exists='append', index=False)
print(f'dim_loc: {len(dim_loc)} registros inseridos')

dim_vei.to_sql('dim_vei', engine, schema='dw', if_exists='append', index=False)
print(f'dim_vei: {len(dim_vei)} registros inseridos')

dim_cli.to_sql('dim_cli', engine, schema='dw', if_exists='append', index=False)
print(f'dim_cli: {len(dim_cli)} registros inseridos')

dim_pag.to_sql('dim_pag', engine, schema='dw', if_exists='append', index=False)
print(f'dim_pag: {len(dim_pag)} registros inseridos')

print('\nTodas as dimensoes foram carregadas com sucesso!')

dim_tmp: 365 registros inseridos
dim_loc: 176 registros inseridos
dim_vei: 7 registros inseridos
dim_cli: 147580 registros inseridos
dim_pag: 5 registros inseridos

Todas as dimensoes foram carregadas com sucesso!


## 12. Recarregar Dimensoes com Chaves

Ler dimensoes do banco para obter as chaves surrogate.

In [131]:
dim_tmp_db = pd.read_sql('SELECT * FROM dw.dim_tmp', conn)
dim_loc_db = pd.read_sql('SELECT * FROM dw.dim_loc', conn)
dim_vei_db = pd.read_sql('SELECT * FROM dw.dim_vei', conn)
dim_cli_db = pd.read_sql('SELECT * FROM dw.dim_cli', conn)
dim_pag_db = pd.read_sql('SELECT * FROM dw.dim_pag', conn)

print('Dimensoes recarregadas com chaves surrogate.')

Dimensoes recarregadas com chaves surrogate.


## 13. Criar Fato Corridas (ft_crr)

Montar tabela fato com FKs para dimensoes.

In [ ]:
ft_crr = df_silver.copy()

ft_crr['date'] = pd.to_datetime(ft_crr['date'])
dim_tmp_db['data'] = pd.to_datetime(dim_tmp_db['data'])

ft_crr = ft_crr.merge(
    dim_tmp_db[['tmp_srk', 'data']],
    left_on='date',
    right_on='data',
    how='left'
).drop('data', axis=1)

ft_crr = ft_crr.merge(
    dim_loc_db[['loc_srk', 'local']],
    left_on='pickup',
    right_on='local',
    how='left'
).rename(columns={'loc_srk': 'loc_ori_srk'}).drop('local', axis=1)

ft_crr = ft_crr.merge(
    dim_loc_db[['loc_srk', 'local']],
    left_on='drop',
    right_on='local',
    how='left'
).rename(columns={'loc_srk': 'loc_dst_srk'}).drop('local', axis=1)

ft_crr = ft_crr.merge(
    dim_vei_db[['vei_srk', 'tipo_veiculo']],
    left_on='vehicle',
    right_on='tipo_veiculo',
    how='left'
).drop('tipo_veiculo', axis=1)

ft_crr = ft_crr.merge(
    dim_cli_db[['cli_srk', 'id_cliente']],
    left_on='customer_id',
    right_on='id_cliente',
    how='left'
).drop('id_cliente', axis=1)

ft_crr = ft_crr.merge(
    dim_pag_db[['pag_srk', 'metodo_pagamento']],
    left_on='payment',
    right_on='metodo_pagamento',
    how='left'
).drop('metodo_pagamento', axis=1)

print(f'Fato Corridas preparado: {len(ft_crr)} registros')

print(f'\nVerificando valores NULL nas chaves estrangeiras...')
print(f"tmp_srk NULL: {ft_crr['tmp_srk'].isna().sum()}")
print(f"loc_ori_srk NULL: {ft_crr['loc_ori_srk'].isna().sum()}")
print(f"loc_dst_srk NULL: {ft_crr['loc_dst_srk'].isna().sum()}")
print(f"vei_srk NULL: {ft_crr['vei_srk'].isna().sum()}")
print(f"cli_srk NULL: {ft_crr['cli_srk'].isna().sum()}")
print(f"pag_srk NULL: {ft_crr['pag_srk'].isna().sum()}")

registros_antes = len(ft_crr)
ft_crr = ft_crr.dropna(subset=['tmp_srk', 'loc_ori_srk', 'loc_dst_srk', 'vei_srk', 'cli_srk', 'pag_srk'])
registros_depois = len(ft_crr)

print(f'\nRegistros removidos por falta de chaves: {registros_antes - registros_depois}')
print(f'Registros validos: {registros_depois}')


Fato Corridas preparado: 148767 registros


## 14. Preparar Colunas do Fato

Selecionar e renomear colunas para o fato.

In [133]:
ft_crr_final = ft_crr[[
    'tmp_srk', 'loc_ori_srk', 'loc_dst_srk', 'vei_srk', 'cli_srk', 'pag_srk',
    'distance', 'vtat', 'value', 'value_per_km',
    'driver_rating', 'customer_rating', 'status', 'distance_category',
    'pickup_region', 'drop_region', 'id'
]].copy()

ft_crr_final.rename(columns={
    'distance': 'distancia_km',
    'vtat': 'duracao_minutos',
    'value': 'valor_corrida',
    'value_per_km': 'valor_por_km',
    'driver_rating': 'avaliacao_motorista',
    'customer_rating': 'avaliacao_cliente',
    'status': 'status_corrida',
    'distance_category': 'categoria_distancia',
    'id': 'id_booking_original'
}, inplace=True)

ft_crr_final['eh_corrida_completa'] = ft_crr_final['status_corrida'] == 'Completed'
ft_crr_final['eh_corrida_cancelada'] = ft_crr_final['status_corrida'].isin(['Cancelled by Driver', 'Cancelled by Customer'])
ft_crr_final['eh_rota_inter_regional'] = ft_crr_final['pickup_region'] != ft_crr_final['drop_region']

ft_crr_final['data_hora_coleta'] = pd.Timestamp.now()

ft_crr_final.drop(['pickup_region', 'drop_region'], axis=1, inplace=True)

print(f'Fato final preparado: {len(ft_crr_final)} registros')
print(f'Colunas: {list(ft_crr_final.columns)}')

Fato final preparado: 148767 registros
Colunas: ['tmp_srk', 'loc_ori_srk', 'loc_dst_srk', 'vei_srk', 'cli_srk', 'pag_srk', 'distancia_km', 'duracao_minutos', 'valor_corrida', 'valor_por_km', 'avaliacao_motorista', 'avaliacao_cliente', 'status_corrida', 'categoria_distancia', 'id_booking_original', 'eh_corrida_completa', 'eh_corrida_cancelada', 'eh_rota_inter_regional', 'data_hora_coleta']


## 15. Inserir Fato no Banco

Carregar tabela fato no schema DW.

In [134]:
ft_crr_final.to_sql('ft_crr', engine, schema='dw', if_exists='append', index=False)

print(f'ft_crr: {len(ft_crr_final)} registros inseridos')
print('\nTabela fato carregada com sucesso!')

DatabaseError: Execution failed on sql 'INSERT INTO dw.ft_crr (tmp_srk, loc_ori_srk, loc_dst_srk, vei_srk, cli_srk, pag_srk, distancia_km, duracao_minutos, valor_corrida, valor_por_km, avaliacao_motorista, avaliacao_cliente, status_corrida, categoria_distancia, id_booking_original, eh_corrida_completa, eh_corrida_cancelada, eh_rota_inter_regional, data_hora_coleta) VALUES (:tmp_srk, :loc_ori_srk, :loc_dst_srk, :vei_srk, :cli_srk, :pag_srk, :distancia_km, :duracao_minutos, :valor_corrida, :valor_por_km, :avaliacao_motorista, :avaliacao_cliente, :status_corrida, :categoria_distancia, :id_booking_original, :eh_corrida_completa, :eh_corrida_cancelada, :eh_rota_inter_regional, :data_hora_coleta)': (psycopg2.errors.NotNullViolation) null value in column "pag_srk" of relation "ft_crr" violates not-null constraint
DETAIL:  Failing row contains (1, 1, 1, 76, 1, 16055, null, null, null, null, null, null, null, No Driver Found, null, f, f, t, 2026-01-26 08:52:46.846272, "CNR5884300").

[SQL: INSE

## 16. Diagnostico Pos-ETL

Verificar contagem de registros em todas as tabelas.

In [ ]:
cur.execute("SELECT COUNT(*) FROM dw.dim_tmp")
print(f'dim_tmp: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_loc")
print(f'dim_loc: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_vei")
print(f'dim_vei: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_cli")
print(f'dim_cli: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.dim_pag")
print(f'dim_pag: {cur.fetchone()[0]:,} registros')

cur.execute("SELECT COUNT(*) FROM dw.ft_crr")
print(f'ft_crr: {cur.fetchone()[0]:,} registros')

print('\nETL Silver → Gold concluido com sucesso!')

## 17. Fechar Conexoes

Encerrar conexoes com o banco de dados.

In [ ]:
cur.close()
conn.close()
engine.dispose()

print('Conexoes fechadas.')